In [2]:
import pandas as pd
from lazypredict.Supervised import LazyClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import roc_auc_score
import os
df_ideo= pd.read_csv('/workspace/4GA.DataScience/data/processed/df_mmxeu.csv')
del df_ideo['lrscale','lrscale_mmx']
def bestclassifier(df, target_column, feature_range):
    """
    Evalúa LazyClassifier con diferentes números de características, guarda el mejor modelo y características en un archivo.

    Args:
        df (pd.DataFrame): DataFrame con los datos.
        target_column (str): Nombre de la columna objetivo.
        feature_range (tuple): Rango de números de características a iterar (inicio, fin).
    """

    results = {}
    X = df.drop(target_column, axis=1)
    y = df[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    best_model = None
    best_accuracy = -float('inf')
    best_auc = -float('inf')
    best_k = 0
    best_features = []

    for k in range(feature_range[0], feature_range[1] + 1):
        selector = SelectKBest(score_func=f_classif, k=k)
        X_train_selected = selector.fit_transform(X_train, y_train)
        X_test_selected = selector.transform(X_test)

        clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
        models, predictions = clf.fit(X_train_selected, X_test_selected, y_train, y_test)
        results[k] = models

        top_model_name = models.index[0]
        top_model_accuracy = models.loc[top_model_name, 'Accuracy']
        top_model_auc = roc_auc_score(y_test, predictions[top_model_name])
        models.loc[top_model_name, 'AUC'] = top_model_auc

        if top_model_auc > best_auc: #modificamos la logica para que use AUC
            best_accuracy = top_model_accuracy
            best_auc = top_model_auc
            best_model = top_model_name
            best_k = k
            best_features = X.columns[selector.get_support()].tolist()

    output_dir = 'C:/Users/Josue/4GA.Datascience/4GA.DataScience/App'
    os.makedirs(output_dir, exist_ok=True)

    with open(os.path.join(output_dir, 'bestclassifier.txt'), 'w') as f:
        f.write(f"Mejor modelo: {best_model}\n")
        f.write(f"Accuracy: {best_accuracy}\n")
        f.write(f"AUC: {best_auc}\n")
        f.write(f"Número de características: {best_k}\n")
        f.write(f"Características: {best_features}\n")

    return results, best_model, best_k, best_features



results, best_model, best_k, best_features = bestclassifier(df_ideo, 'ideologia', (10, 18))

print(f"Mejor modelo: {best_model}")
print(f"Número de características: {best_k}")
print(f"Características: {best_features}")

ModuleNotFoundError: No module named 'pandas'